<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">DISTRIBUTED LORA FINE-TUNING WITH LLAMAFACTORY (NODE)</h2>

<p>LoRA is a lightweight fine-tuning method for large language models that injects small, trainable adapter layers into the model instead of updating all parameters. This makes training far more memory and compute-efficient, while reducing the risk of significant forgetting compared to full fine-tuning. However, LoRA adapters may sometimes trade off a bit of peak accuracy for efficiency and flexibility.</p>

<h3 style="color:#44D62C; text-align:left;">📥 1. Download the Model</h3>

Before running a model in distributed mode, ensure it is downloaded on every participating machine.

This command will pull the model from Hugging Face into the local Razer environment cache.

In [ ]:
rzr-aikit model download Qwen/Qwen2.5-0.5B-Instruct

<h3 style="color:#44D62C; text-align:left;">🔗 2. Join Cluster on Worker Node</h3>

On each worker node, you'll need to connect to the cluster initiated by the head node.

This step will register the current machine as a worker, allowing it to participate in distributed model serving.

Use the <code>--ifname</code> flag to specify the correct network interface (e.g. <code>enp60s0</code>), and <code>--address</code> to point to the head node's IP and cluster port.

In [ ]:
ip -f inet -4 -o addr

In [ ]:

rzr-aikit cluster join --ifname enp60s0 --address 192.168.8.4:6379 --metrics-export-port 8080

<h3 style="color:#44D62C; text-align:left;">🛰️ 3. Check Cluster Status</h3>

After joining the cluster from a worker node, you can verify whether the node has successfully registered with the head.

This command will show a list of connected nodes, their roles (head/worker), and available resources.

In [ ]:
rzr-aikit cluster status

<h3 style="color:#44D62C; text-align:left;">📊 4. Dataset Configuration</h3>

LlamaFactory uses JSON configuration files to define dataset parameters, including data sources, formatting templates, and column mappings. This approach provides flexibility in handling various dataset formats and ensures consistent data preprocessing across distributed training.

In [ ]:
# Create data directory and dataset configuration
mkdir -p /var/aikit/fine-tuning/data
cat > /var/aikit/fine-tuning/data/dataset_info.json << 'JSON'
{
  "alpaca_gpt4": {
    "hf_hub_url": "vicgalle/alpaca-gpt4",
    "formatting": "alpaca",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}
JSON

<h3 style="color:#44D62C; text-align:left;">🛑 5. Cleanup</h3>

Once you're finished with distributed fine-tuning, you can gracefully shut down the cluster on both the head and worker nodes to free up GPU and system resources.

In [ ]:
rzr-aikit cluster stop